# Incremental file processing

Use `magie.file_sync.sync_files` with three functions: find source files, map
each source to its destination, and perform the conversion. This self-contained
example uses temporary text files, so no archive data or network is needed.
Run with a Python environment containing MAGIE.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from magie.file_sync import sync_files

workspace = TemporaryDirectory(prefix="magie-sync-tutorial-")
root = Path(workspace.name)
source_root = root / "source"
output_root = root / "output"
source_root.mkdir()
(source_root / "example.txt").write_text("hello from MAGIE", encoding="utf-8")


## 1. Supply the three functions

The mapper receives the full absolute source `Path`. Here `relative_to` keeps
its path under the source root, and `with_suffix` changes the extension.
The processor writes directly to the exact final destination; the helper
creates its parent directory. No temporary output files are used. A failed
write can leave partial output, which is retried on the next run.


In [ ]:
def source_files():
    return source_root.rglob("*.txt")


def destination_for(source):
    return (output_root / source.relative_to(source_root)).with_suffix(".out")


def process(source, destination):
    text = source.read_text(encoding="utf-8")
    destination.write_text(text.upper(), encoding="utf-8")


def run_sync(*, dry_run=False):
    return sync_files(
        source_files,
        destination_for,
        process,
        state_path=output_root / ".sync.sqlite3",
        job="uppercase-v1",
        dry_run=dry_run,
    )


## 2. Preview, then run

The dry run reports pending updates without creating outputs or state.


In [ ]:
preview = run_sync(dry_run=True)
assert preview.updated == 1 and not output_root.exists()
print(preview)

first = run_sync()
assert first.updated == 1 and first.failed == 0
print(first)
print((output_root / "example.out").read_text(encoding="utf-8"))


## 3. Skip unchanged files and pick up changes

Discovery and destination mapping run every time. Processing is skipped when
both source and output sizes and modification times match saved records.
Older files arriving later are found too, provided your discovery function
includes them. Missing or changed outputs are also regenerated.


In [ ]:
unchanged = run_sync()
assert unchanged.skipped == 1 and unchanged.updated == 0
print(unchanged)

(source_root / "example.txt").write_text("updated input", encoding="utf-8")
(source_root / "late-arrival.txt").write_text("arrived later", encoding="utf-8")
changed = run_sync()
assert changed.updated == 2 and changed.failed == 0
print(changed)


For another job, replace the three functions. For IAGA output your processor
can use MagPy's reader and writer; validate the stream metadata in that function.

Keep a stable state file and job name between runs. Change `job` when processing
code or settings change and you want a rebuild. Files without state are processed
once, even when the destination already exists. Per-file failures are logged and
counted in `result.failed`; successes stay recorded and failures retry next run.

This interface processes one source into one output. Changes preserving both
size and modification time are not detected. Removed sources do not delete
outputs. Use one state file for jobs writing the same outputs; overlapping
normal runs are rejected by its POSIX file lock.

Live sources may change during processing. Successful processing still counts
as an update; the saved source metadata comes from before processing, so the
next run picks up the change. Actual processor errors still count as failures.


In [ ]:
# Remove only this tutorial's temporary workspace when finished.
workspace.cleanup()
